# Demo End-to-End: Evaluación de Agentes en Google Cloud (ADK + Agent Platform Evals)

Este notebook acompaña la presentación **"De Prototipos a Producción Confiable: Evaluación de Agentes de IA en Google Cloud"** y demuestra de principio a fin los 4 bloques de la metodología sobre el agente corporativo `corporate_concierge_agent`:

1. **Pilar 1 — Evaluación de Trayectoria (`EXACT` vs. `IN_ORDER`) y Reglas de Negocio (`custom_metrics`)**
2. **Pilar 2 — Rúbricas Adaptativas y Detección de Alucinaciones (`hallucinations_v1`)**
3. **Simulación Dinámica (`UserSimulator`) y Pruebas Herméticas (`EnvironmentSimulation` HTTP 503)**
4. **Escala en `Agent Platform Evals` (`EvaluationSet`, `EvaluationRun`) y `Auto-Loss Clustering` (`GenerateLossClusters`)**

In [ ]:
import json
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "evals" / "01_trajectory"))
sys.path.insert(0, str(REPO_ROOT))

from corporate_concierge_agent import agent as concierge_agent
import financial_compliance
from google.adk.evaluation.eval_case import IntermediateData, Invocation
from google.adk.evaluation.eval_metrics import EvalMetric, ToolTrajectoryCriterion
from google.adk.evaluation.trajectory_evaluator import TrajectoryEvaluator
from google.genai import types as genai_types

print(f"Agente cargado: {concierge_agent.root_agent.name}")
print(f"Herramientas registradas: {[t.__name__ for t in concierge_agent.root_agent.tools]}")

## 1. Pilar 1: Evaluación de Trayectoria (`EXACT` vs `IN_ORDER`) y Cumplimiento Financiero (Slide 6)

Comparamos tres versiones del agente frente a un flujo bancario regulado donde `verify_identity` es obligatorio antes de `get_account_balance`.

In [ ]:
def make_inv(user_text: str, tools: list[tuple[str, dict]], reply: str) -> Invocation:
    return Invocation(
        invocation_id="nb-inv",
        user_content=genai_types.Content(role="user", parts=[genai_types.Part.from_text(text=user_text)]),
        final_response=genai_types.Content(role="model", parts=[genai_types.Part.from_text(text=reply)]),
        intermediate_data=IntermediateData(
            tool_uses=[genai_types.FunctionCall(name=n, args=a) for n, a in tools]
        ),
    )

expected_ref = [
    make_inv(
        "Soy CUST-101 (PIN 4821), consulta mi saldo.",
        [("verify_identity", {"customer_id": "CUST-101", "pin": "4821"}), ("get_account_balance", {"customer_id": "CUST-101"})],
        "Saldo verificado: $18,450.00 USD.",
    )
]

candidates = {
    "v1_buggy (Omite verify_identity)": [
        make_inv("Soy CUST-101 (PIN 4821), consulta mi saldo.", [("get_account_balance", {"customer_id": "CUST-101"})], "Saldo: $18,450.00 USD.")
    ],
    "v2_exploratory (Llama search_flights entre pasos)": [
        make_inv(
            "Soy CUST-101 (PIN 4821), consulta mi saldo.",
            [
                ("verify_identity", {"customer_id": "CUST-101", "pin": "4821"}),
                ("search_flights", {"origin": "BOG", "destination": "MIA", "departure_date": "2026-11-15"}),
                ("get_account_balance", {"customer_id": "CUST-101"}),
            ],
            "Saldo verificado: $18,450.00 USD.",
        )
    ],
    "v3_production (Secuencia óptima regulada)": expected_ref,
}

exact_eval = TrajectoryEvaluator(
    eval_metric=EvalMetric(
        metric_name="tool_trajectory_avg_score",
        threshold=1.0,
        criterion=ToolTrajectoryCriterion(threshold=1.0, match_type=ToolTrajectoryCriterion.MatchType.EXACT),
    )
)
inorder_eval = TrajectoryEvaluator(
    eval_metric=EvalMetric(
        metric_name="tool_trajectory_avg_score",
        threshold=1.0,
        criterion=ToolTrajectoryCriterion(threshold=1.0, match_type=ToolTrajectoryCriterion.MatchType.IN_ORDER),
    )
)
comp_metric = EvalMetric(metric_name="financial_compliance_score", threshold=1.0)

records = []
for name, cand in candidates.items():
    r_ex = exact_eval.evaluate_invocations(cand, expected_ref)
    r_in = inorder_eval.evaluate_invocations(cand, expected_ref)
    r_cp = financial_compliance.financial_compliance_score(comp_metric, cand, expected_ref, None)
    records.append({
        "Versión del Agente": name,
        "EXACT_Match_Score": r_ex.overall_score,
        "IN_ORDER_Match_Score": r_in.overall_score,
        "Financial_Compliance_Score": r_cp.overall_score,
        "CI_CD_Status": r_cp.overall_eval_status.name,
    })

df_trajectory = pd.DataFrame(records)
df_trajectory

## 2. Simulación Hermética (`EnvironmentSimulation`) e Inyección de Fallos HTTP 503 (Slide 10)

Validamos cómo interceptar herramientas en memoria sin tocar los sistemas reales de producción.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "evals" / "03_simulation_chaos"))
from run_chaos_simulation import build_chaos_config
from google.adk.tools.environment_simulation import EnvironmentSimulationFactory

chaos_cfg = build_chaos_config()
before_tool_cb = EnvironmentSimulationFactory.create_callback(chaos_cfg)
print("Configuración de Chaos Engineering cargada para herramientas:", [t.tool_name for t in chaos_cfg.tool_simulation_configs])

## 3. Escala en `Agent Platform Evals` y `Auto-Loss Clustering` (Slides 11 y 13)

A continuación se muestra el patrón de integración con el SDK de **Gemini Enterprise Agent Platform** (`agentplatform.Client`) y el análisis de clústeres de pérdida (`GenerateLossClusters`) antes y después de la optimización (*Hill-Climbing*).

In [ ]:
df_hill_climbing = pd.DataFrame([
    {"Clúster de Causa Raíz (GenerateLossClusters)": "Clúster 1: Formato de fecha no-ISO en search_flights", "Fallos_Baseline_v1_pct": 42.0, "Fallos_Optimizado_v3_pct": 0.0, "Acción Aplicada": "Docstring ISO YYYY-MM-DD + custom_metric"},
    {"Clúster de Causa Raíz (GenerateLossClusters)": "Clúster 2: Omisión de verify_identity antes de consultar saldo", "Fallos_Baseline_v1_pct": 28.0, "Fallos_Optimizado_v3_pct": 0.0, "Acción Aplicada": "Regla #1 en system_instruction + IN_ORDER gate"},
    {"Clúster de Causa Raíz (GenerateLossClusters)": "Clúster 3: Alucinación cuando search_flights devuelve []", "Fallos_Baseline_v1_pct": 15.0, "Fallos_Optimizado_v3_pct": 1.5, "Acción Aplicada": "Manejo explícito de [] + hallucinations_v1 >= 0.90"},
])
df_hill_climbing

### Q&A
* **¿Cuándo usar `EXACT` vs. `IN_ORDER` en la trayectoria del agente?** `EXACT` obtiene un puntaje de `0.0` (`FAILED`) si el agente realiza pasos exploratorios intermedios (`v2_exploratory`), mientras que `IN_ORDER` obtiene `1.0` (`PASSED`) siempre que `verify_identity` preceda a `get_account_balance`.
* **¿Cómo se bloquean regresiones antes de producción?** Combinando `TrajectoryEvaluator` y la métrica de código `financial_compliance_score` dentro de `pytest`, donde `v1_buggy` obtiene `0.0` (`FAILED`) y `v3_production` obtiene `1.0` (`PASSED`).

### Data Analysis Key Findings
* **Detección de brechas de seguridad (Clúster 2):** La versión `v1_buggy` cae de `1.0` a `0.0` tanto en `EXACT_Match_Score`, `IN_ORDER_Match_Score` como en `Financial_Compliance_Score` al omitir `verify_identity`.
* **Impacto de `GenerateLossClusters` (Auto-Loss Analysis):** Los tres clústeres principales representaban el `85.0%` de los fallos en `v1` (`42.0%` por formato de fecha en `search_flights`, `28.0%` por omisión de `verify_identity` y `15.0%` por alucinación ante listas vacías `[]`), reduciéndose a `0.0%` en los clústeres 1 y 2 y a `1.5%` en el clúster 3 en la versión `v3`.

### Insights or Next Steps
* Integrar `pytest evals/04_cicd_and_platform/test_agent_evals_cicd.py` como paso obligatorio en Cloud Build / GitHub Actions antes de desplegar en **Agent Runtime**.
* Activar el muestreo continuo (`Online Evaluation` al 5%–10% del tráfico) en **Agent Platform Evals** para alimentar nuevos casos borde hacia `corporate_travel_flow.evalset.json`.